# NyayaRAG Colab Artifact Builder

Run this notebook on a Colab GPU. It only builds `nyayarag_artifacts.zip` in Google Drive. Download that zip and unzip it locally to run Streamlit inference.

In [ ]:
!nvidia-smi

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.environ["HOME"] + "/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
%cd /content
!rm -rf NyayaRAG
!git clone https://github.com/mi-bilal/NyayaRAG.git
%cd /content/NyayaRAG

## Configure HF Token

Option A: add `HF_TOKEN` in Colab secrets and run the next cell. Option B: replace `your_hf_token_here` manually in the fallback cell. Groq is not needed for artifact building.

In [ ]:
from google.colab import userdata

hf = userdata.get("HF_TOKEN") or ""
env = f"""HF_TOKEN={hf}
GROQ_MODEL=openai/gpt-oss-120b
EMBEDDING_MODEL=Qwen/Qwen3-Embedding-0.6B
EMBEDDING_DIM=1024
QDRANT_PATH=artifacts/qdrant
QDRANT_COLLECTION=nyayarag_statutes
BM25_PATH=artifacts/bm25/statutes_bm25.pkl
DATA_DIR=data
ARTIFACT_DIR=artifacts
TOP_K=5
CANDIDATE_POOL=80
RRF_K=60
MAX_CASE_TOKENS=3500
MAX_CONTEXT_TOKENS=5000
"""
open(".env", "w").write(env)
print("Wrote .env; HF token present:", bool(hf))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!uv sync --dev
!uv run python -c "import accelerate; print('accelerate', accelerate.__version__)"

## Small Test Build

Run this first. It builds a small zip to prove everything works.

In [ ]:
!rm -rf artifacts
!uv run python scripts/colab_full_build.py --batch-size 8 --embed-max-tokens 512 --embed-overlap-tokens 64 --limit 500

## Full Build

Run this after the small test succeeds. It may take a while. The output zip is written to Drive.

In [ ]:
!rm -rf artifacts
!uv run python scripts/colab_full_build.py --batch-size 8 --embed-max-tokens 512 --embed-overlap-tokens 64

In [ ]:
!ls -lh /content/drive/MyDrive/nyayarag_artifacts/

Download `/content/drive/MyDrive/nyayarag_artifacts/nyayarag_artifacts.zip`, unzip it into your local repo root, then run `uv run streamlit run app/streamlit_app.py`.